# Track A3: Data Profiling Visualizations

This notebook visualizes key statistical distributions from the Track A3 profiling tables:
1. **Description Length Distribution & Quantiles** (Characters & Words, Right-Skewness, Temporal Stability)
2. **HS Code Digit-Length Distribution** (Systematic Leading-Zero Truncation Check)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set clean visual style
sns.set_theme(style="whitegrid", font_scale=1.05)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TABLES_DIR = ROOT / 'reports' / 'tables'
print(f"Reading profiling tables from: {TABLES_DIR}")

## 1. Description Length Distribution (Characters & Words)
- **Right-Skewness Assessment:** Compares minimum, median, 95th percentile, and maximum lengths.
- **Sequence Length Impact:** Visualizes P95 cutoffs for NLP tokenizer sequence length planning.

In [ ]:
df_len = pd.read_csv(TABLES_DIR / 'a3_description_length_quantiles.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character Lengths
years = df_len['year'].astype(str)
x_indices = range(len(years))
width = 0.25

axes[0].bar([x - width for x in x_indices], df_len['approx_median_chars'], width=width, label='Median (Approx)', color='#2b5c8f')
axes[0].bar(x_indices, df_len['approx_p95_chars'], width=width, label='95th Percentile (Approx)', color='#e27d60')
axes[0].bar([x + width for x in x_indices], df_len['max_chars'], width=width, label='Max (256-char cap)', color='#c38d9e')

axes[0].set_title('Goods Description Character Lengths by Year')
axes[0].set_xlabel('Declaration Year')
axes[0].set_ylabel('Character Count')
axes[0].set_xticks(x_indices)
axes[0].set_xticklabels(years)
axes[0].legend()

# Word Counts
axes[1].bar([x - width/2 for x in x_indices], df_len['approx_median_words'], width=width, label='Median Words (Approx)', color='#41b3a3')
axes[1].bar([x + width/2 for x in x_indices], df_len['approx_p95_words'], width=width, label='95th Percentile Words (Approx)', color='#e8a87c')

axes[1].set_title('Goods Description Word Counts by Year')
axes[1].set_xlabel('Declaration Year')
axes[1].set_ylabel('Word Count')
axes[1].set_xticks(x_indices)
axes[1].set_xticklabels(years)
axes[1].legend()

plt.tight_layout()
plt.show()

### Statistical Notes:
- **Strong Positive Skewness:** Median character length is ~45 chars (~7 words), but P95 reaches 125 chars (~19 words).
- **Max Length Plateau:** All three years peak at exactly 256 characters, pointing to a hard database field restriction during customs logging.
- **Short Token Share:** Between 10.3% and 11.4% of descriptions contain $\le 2$ words.

## 2. HS Code Digit-Length Distribution
- **Systematic Truncation Check:** Verifies whether length variations are random user errors or systematic leading zero omission on Chapters 01–09.

In [ ]:
df_digits = pd.read_csv(TABLES_DIR / 'a3_code_digit_distribution.csv')

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=df_digits, 
    x='year', 
    y='pct_of_year', 
    hue='digit_length', 
    palette='muted',
    ax=ax
)

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{height:.2f}%", 
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10, xytext=(0, 3), 
                    textcoords='offset points')

ax.set_title('HS Code Length Distribution: 10 vs 11 Digits')
ax.set_xlabel('Declaration Year')
ax.set_ylabel('Percentage of Year (%)')
ax.set_ylim(0, 115)
ax.legend(title='Digit Length')

plt.tight_layout()
plt.show()

### Statistical Notes:
- **Clean Bimodal Split:** Exactly ~98.5% of transactions are 11 digits, and ~1.5% are 10 digits across all years.
- **Zero Random Digit Noise:** There are zero records with other lengths (e.g., 6, 8, 9, 12 digits), confirming that this is not human typo noise, but solely dropped leading zeros on Chapters 01–09 from the original `BIGINT` database type.
- **Remediation Plan:** In Track A4, left-padding 10-digit codes with a single `'0'` completely resolves 100% of the length variation.